In [2]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# 1. Gün temizlediğimiz veriyi yüklüyoruz
df = pd.read_csv('../data/processed/cleaned_retail.csv')
print(f"Toplam Satır: {df.shape[0]}")
df.head()

Toplam Satır: 407664


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.9500,13085,United Kingdom,83.4000
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.7500,13085,United Kingdom,81.0000
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.7500,13085,United Kingdom,81.0000
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.1000,13085,United Kingdom,100.8000
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.2500,13085,United Kingdom,30.0000


In [3]:
# Sadece Birleşik Krallık siparişlerini filtreleyelim
df_uk = df[df['Country'] == 'United Kingdom']

# Fatura ve Ürün bazında gruplayıp miktar tablosunu oluşturalım
uk_basket = df_uk.groupby(['Invoice', 'Description'])['Quantity'].sum().unstack().fillna(0)

# Miktarları 1 (sepette var) ve 0 (sepette yok) şeklinde bool/binary formata çevirelim
uk_basket_matrix = uk_basket.map(lambda x: 1 if x > 0 else 0).astype(bool)

print(f"Sepet Matrisi Boyutu: {uk_basket_matrix.shape[0]} Fatura, {uk_basket_matrix.shape[1]} Tekil Ürün")
uk_basket_matrix.iloc[0:5, 0:5]

Sepet Matrisi Boyutu: 17612 Fatura, 4415 Tekil Ürün


Description,DOORMAT UNION JACK GUNS AND ROSES,3 STRIPEY MICE FELTCRAFT,4 PURPLE FLOCK DINNER CANDLES,ANIMAL STICKERS,BLACK PIRATE TREASURE CHEST
Invoice,,,,,
489434,False,False,False,False,False
489435,False,False,False,False,False
489436,False,False,False,False,False
489437,False,False,False,False,False
489438,False,False,False,False,False


In [4]:
frequent_itemsets = apriori(uk_basket_matrix, min_support=0.01, use_colnames=True)
frequent_itemsets = frequent_itemsets.sort_values(by='support', ascending=False)

print(f"Tespit Edilen Sık Ürün Seti Sayısı: {len(frequent_itemsets)}")
frequent_itemsets.head(10)

Tespit Edilen Sık Ürün Seti Sayısı: 822


,support,itemsets
548,0.1658,(WHITE HANGING HEART T-LIGHT HOLDER)
411,0.0860,(REGENCY CAKESTAND 3 TIER)
30,0.0715,(ASSORTED COLOUR BIRD ORNAMENT)
500,0.0703,(STRAWBERRY CERAMIC TRINKET BOX)
220,0.0646,(HOME BUILDING BLOCK WORD)
422,0.0588,(REX CASH+CARRY JUMBO SHOPPER)
18,0.0580,(60 TEATIME FAIRY CAKE CASES)
325,0.0567,(PACK OF 72 RETRO SPOT CAKE CASES)
240,0.0562,(JUMBO BAG RED RETROSPOT)
218,0.0559,(HEART OF WICKER LARGE)


In [5]:
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

# En yüksek birliktelik gücüne (Lift) göre sıralayalım
rules_sorted = rules.sort_values(by=['lift', 'confidence'], ascending=[False, False])

# İlgili metrik sütunlarını filtreleyip ilk 10 kuralı gösterelim
rules_summary = rules_sorted[['antecedents', 'consequents', 'support', 'confidence', 'lift', 'leverage', 'conviction']]
rules_summary.head(10)


,antecedents,consequents,support,confidence,lift,leverage,conviction
414,(POPPY'S PLAYHOUSE KITCHEN),(POPPY'S PLAYHOUSE LIVINGROOM ),0.0108,0.7100,61.2999,0.0107,3.4088
415,(POPPY'S PLAYHOUSE LIVINGROOM ),(POPPY'S PLAYHOUSE KITCHEN),0.0108,0.9363,61.2999,0.0107,15.4526
279,(POPPY'S PLAYHOUSE BEDROOM ),(POPPY'S PLAYHOUSE KITCHEN),0.0123,0.9042,59.1977,0.0121,10.2754
278,(POPPY'S PLAYHOUSE KITCHEN),(POPPY'S PLAYHOUSE BEDROOM ),0.0123,0.8067,59.1977,0.0121,5.1026
378,(GREEN REGENCY TEACUP AND SAUCER),(ROSES REGENCY TEACUP AND SAUCER ),0.0110,0.8255,55.7060,0.0108,5.6468
379,(ROSES REGENCY TEACUP AND SAUCER ),(GREEN REGENCY TEACUP AND SAUCER),0.0110,0.7433,55.7060,0.0108,3.8435
374,(COFFEE MUG DOG + BALL DESIGN),(COFFEE MUG CAT + BIRD DESIGN),0.0111,0.7471,50.0319,0.0109,3.8955
375,(COFFEE MUG CAT + BIRD DESIGN),(COFFEE MUG DOG + BALL DESIGN),0.0111,0.7414,50.0319,0.0109,3.8103
302,(SET/10 PINK SPOTTY PARTY CANDLES),(SET/10 BLUE SPOTTY PARTY CANDLES),0.0120,0.6741,46.3774,0.0117,3.0240
303,(SET/10 BLUE SPOTTY PARTY CANDLES),(SET/10 PINK SPOTTY PARTY CANDLES),0.0120,0.8242,46.3774,0.0117,5.5878


In [6]:
import os

output_path = os.path.abspath("../data/processed/association_rules.csv")
rules_sorted.to_csv(output_path, index=False)
print(f"Birliktelik kuralları başarıyla kaydedildi: {output_path}")

Birliktelik kuralları başarıyla kaydedildi: C:\Users\USER\ecommerce-intelligence\data\processed\association_rules.csv
